<a href="https://colab.research.google.com/github/EgzonnOsmanaj/MesoAI/blob/main/eu_ai_actRAG_WBS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>



# EU AI Act RAG System: Advanced Hybrid Retrieval & Generation

## 1. Domain & Data Sourcing Justification
**Domain:** European Union AI Regulatory Text (Regulation (EU) 2024/1689).
**Dataset:** The official 144-page PDF of the EU AI Act published in the Official Journal of the European Union.

**Why this domain requires RAG:** The EU AI Act is a dense, highly specialized legal framework with strict compliance timelines, nuanced risk categorizations, and specific exclusions. Standard LLMs struggle significantly with this domain because:
1. **Hallucination Risk:** They tend to hallucinate specific legal articles, exact penalty amounts, and enforcement dates.
2. **Context Window Limitations:** The full legal text exceeds standard context windows for detailed, multi-document cross-referencing.
3. **Verification:** Legal and compliance professionals require exact page citations and article references to verify claims, which a standard LLM cannot reliably provide without a grounded RAG pipeline.



In [ ]:
!pip -q install pandas numpy scikit-learn sentence-transformers faiss-cpu rank-bm25 pymupdf requests beautifulsoup4 lxml google-genai tqdm

In [ ]:
import os
import re
import json
import time
import requests
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from dataclasses import dataclass
import fitz
import faiss
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer, CrossEncoder
from google import genai

In [ ]:
DATA_DIR = "data"
RAW_DIR = os.path.join(DATA_DIR, "raw")
PROCESSED_DIR = os.path.join(DATA_DIR, "processed")
EVAL_DIR = os.path.join(DATA_DIR, "eval")
for d in [DATA_DIR, RAW_DIR, PROCESSED_DIR, EVAL_DIR]:
    os.makedirs(d, exist_ok=True)

PDF_PATH = os.path.join(RAW_DIR, "eu_ai_act.pdf") # Added this line
PDF_URL = "https://eur-lex.europa.eu/legal-content/EN/TXT/PDF/?uri=OJ:L_202401689"

if not os.path.exists(PDF_PATH):
    print(f"Downloading PDF from {PDF_URL} to {PDF_PATH}")
    response = requests.get(PDF_URL)
    response.raise_for_status() # Raise an exception for bad status codes
    with open(PDF_PATH, "wb") as f:
        f.write(response.content)
    print("Download complete.")

GEMINI_API_KEY = os.environ.get("GEMINI_API_KEY", "")
GEMINI_MODEL = "gemini-2.5-flash"

EMBED_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
RERANK_MODEL_NAME = "cross-encoder/ms-marco-MiniLM-L-6-v2"

TOP_K_BM25 = 20
TOP_K_DENSE = 20
TOP_K_RERANK = 5

In [ ]:
import os
from google.colab import userdata

# Fetch the Gemini API key from the Colab secrets manager
GEMINI_API_KEY = userdata.get('GoogleAIAPI')

if not GEMINI_API_KEY:
    raise ValueError("Please set GEMINI_API_KEY in your environment or Colab secrets (named 'GoogleAIAPI') first.")

client = genai.Client(api_key=GEMINI_API_KEY)

In [ ]:
@dataclass
class Chunk:
    chunk_id: str
    doc_id: str
    text: str
    metadata: dict

def clean_text(text):
    text = text.replace("\r", "\n")
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()

def split_long_text(text, max_chars=1800, overlap=200):
    text = clean_text(text)
    if len(text) <= max_chars:
        return [text]
    parts = []
    start = 0
    while start < len(text):
        end = min(start + max_chars, len(text))
        parts.append(text[start:end].strip())
        if end == len(text):
            break
        start = end - overlap
    return [p for p in parts if p]

In [ ]:
def extract_pdf_pages(pdf_path):
    doc = fitz.open(pdf_path)
    rows = []
    for i in range(len(doc)):
        page = doc.load_page(i)
        text = clean_text(page.get_text("text"))
        if text:
            rows.append({
                "doc_id": "eu_ai_act",
                "page": i + 1,
                "title": "EU AI Act",
                "text": text
            })
    return pd.DataFrame(rows)


pages_df = extract_pdf_pages(PDF_PATH)
pages_df.head()

In [ ]:
def build_chunks_from_pdf_pages(pages_df):
    chunks = []
    for _, row in pages_df.iterrows():
        parts = split_long_text(row["text"], max_chars=1800, overlap=200)
        for j, part in enumerate(parts):
            chunks.append(
                Chunk(
                    chunk_id=f"{row['doc_id']}_page_{int(row['page'])}_{j}",
                    doc_id=row["doc_id"],
                    text=part,
                    metadata={
                        "title": row["title"],
                        "page": int(row["page"]),
                        "part": j
                    }
                )
            )
    return chunks

chunks = build_chunks_from_pdf_pages(pages_df)
len(chunks), chunks[0]

In [ ]:
def chunks_to_dataframe(chunks):
    rows = []
    for c in chunks:
        row = {
            "chunk_id": c.chunk_id,
            "doc_id": c.doc_id,
            "text": c.text
        }
        row.update(c.metadata)
        rows.append(row)
    return pd.DataFrame(rows)

chunks_df = chunks_to_dataframe(chunks)
chunks_df.to_csv(os.path.join(PROCESSED_DIR, "chunks.csv"), index=False)
chunks_df.head()

In [ ]:
embed_model = SentenceTransformer(EMBED_MODEL_NAME)

chunk_texts = [c.text for c in chunks]
chunk_embeddings = embed_model.encode(
    chunk_texts,
    normalize_embeddings=True,
    show_progress_bar=True
).astype("float32")

dim = chunk_embeddings.shape[1]
faiss_index = faiss.IndexFlatIP(dim)
faiss_index.add(chunk_embeddings)

print("Chunks:", len(chunks))
print("Embedding dim:", dim)
print("FAISS size:", faiss_index.ntotal)

In [ ]:
tokenized_corpus = [c.text.lower().split() for c in chunks]
bm25 = BM25Okapi(tokenized_corpus)

In [ ]:
reranker = CrossEncoder(RERANK_MODEL_NAME)

In [ ]:
def dense_retrieve(query, top_k=10):
    q_emb = embed_model.encode([query], normalize_embeddings=True).astype("float32")
    scores, idxs = faiss_index.search(q_emb, top_k)
    results = []
    for score, idx in zip(scores[0], idxs[0]):
        c = chunks[idx]
        results.append({"chunk": c, "score": float(score), "method": "dense"})
    return results

def bm25_retrieve(query, top_k=10):
    scores = bm25.get_scores(query.lower().split())
    idxs = np.argsort(scores)[::-1][:top_k]
    results = []
    for idx in idxs:
        c = chunks[idx]
        results.append({"chunk": c, "score": float(scores[idx]), "method": "bm25"})
    return results

def reciprocal_rank_fusion(result_lists, k=60):
    fused = {}
    chunk_map = {}
    for res_list in result_lists:
        for rank, item in enumerate(res_list, start=1):
            cid = item["chunk"].chunk_id
            fused[cid] = fused.get(cid, 0.0) + 1.0 / (k + rank)
            chunk_map[cid] = item["chunk"]
    merged = [{"chunk": chunk_map[cid], "score": score} for cid, score in fused.items()]
    merged.sort(key=lambda x: x["score"], reverse=True)
    return merged

def rerank(query, candidates, top_k=5):
    pairs = [(query, c["chunk"].text) for c in candidates]
    scores = reranker.predict(pairs)
    ranked = sorted(zip(candidates, scores), key=lambda x: x[1], reverse=True)
    return [{"chunk": item[0]["chunk"], "score": float(item[1])} for item in ranked[:top_k]]

In [ ]:
def rewrite_query(query):
    q = query.strip()
    q = re.sub(r"\bAI Act\b", "Regulation (EU) 2024/1689", q, flags=re.I)
    q = re.sub(r"\bhigh risk\b", "high-risk AI system", q, flags=re.I)
    return q

In [ ]:
def generate_with_gemini(prompt, model=GEMINI_MODEL):
    response = client.models.generate_content(
        model=model,
        contents=prompt
    )
    return response.text

In [ ]:
def build_prompt(query, retrieved_chunks):
    context = "\n\n".join([
        f"[{i+1}] Page {c.metadata.get('page','?')}:\n{c.text}"
        for i, c in enumerate(retrieved_chunks)
    ])
    return f"""
You are an expert EU legal assistant.
Answer only from the provided context.
If the answer is not supported, say you do not have enough evidence.
Cite the page numbers where relevant.

Question:
{query}

Context:
{context}

Answer:
""".strip()

In [ ]:
def baseline_answer(query, top_k=5):
    dense_hits = dense_retrieve(query, top_k=TOP_K_DENSE)
    top_chunks = [x["chunk"] for x in dense_hits[:top_k]]
    prompt = build_prompt(query, top_chunks)
    answer = generate_with_gemini(prompt)
    return {
        "query": query,
        "rewritten_query": query,
        "retrieved": top_chunks,
        "answer": answer
    }

In [ ]:
def enhanced_answer(query, top_k=5):
    q = rewrite_query(query)

    bm25_hits = bm25_retrieve(q, top_k=TOP_K_BM25)
    dense_hits = dense_retrieve(q, top_k=TOP_K_DENSE)

    fused = reciprocal_rank_fusion([bm25_hits, dense_hits])
    top_for_rerank = fused[:20]

    reranked = rerank(q, top_for_rerank, top_k=top_k)
    top_chunks = [x["chunk"] for x in reranked]

    prompt = build_prompt(query, top_chunks)
    answer = generate_with_gemini(prompt)

    return {
        "query": query,
        "rewritten_query": q,
        "retrieved": top_chunks,
        "answer": answer
    }

In [ ]:
test_queries = [
    "What is the purpose of the AI Act?",
    "When do the transparency rules start to apply?",
    "What are high-risk AI systems?",
    "Which AI practices are prohibited?",
    "What obligations apply to providers of GPAI models?"
]

In [ ]:
for q in test_queries[:3]:
    print("=" * 120)
    print("QUESTION:", q)
    try:
        res = enhanced_answer(q)
        print("\nANSWER:\n", res["answer"])
        print("\nRETRIEVED PAGES:", [c.metadata.get("page") for c in res["retrieved"]])
    except Exception as e:
        print("Error:", e)

In [ ]:
eval_data = [
    {
        "query": "What is the purpose of the AI Act?",
        "gold_answer": "The AI Act aims to regulate AI systems according to risk and support trustworthy AI while protecting fundamental rights.",
        "gold_pages": [1, 2, 3]
    },
    {
        "query": "A researcher develops an AI model exclusively for scientific research purposes and does not place it on the market. Is the model covered by the AI Act?",
        "gold_answer": "No. AI systems and models developed exclusively for scientific research and development are excluded from the scope of the AI Act. Research, testing, and development activities conducted before a system is placed on the market or put into service are also excluded.",
        "gold_pages": [46]
    },
    {
        "query": "What AI practices are prohibited?",
        "gold_answer": "Prohibited practices are those classified as unacceptable risk under the Act.",
        "gold_pages": [51, 52]
    }
]

eval_df = pd.DataFrame(eval_data)
eval_df.to_csv(os.path.join(EVAL_DIR, "eval_questions.csv"), index=False)
eval_df

In [ ]:
def retrieve_pages(query, method="enhanced", top_k=5):
    if method == "baseline":
        hits = dense_retrieve(query, top_k=top_k)
        return [h["chunk"].metadata.get("page") for h in hits]
    q = rewrite_query(query)
    bm25_hits = bm25_retrieve(q, top_k=TOP_K_BM25)
    dense_hits = dense_retrieve(q, top_k=TOP_K_DENSE)
    fused = reciprocal_rank_fusion([bm25_hits, dense_hits])
    reranked = rerank(q, fused[:20], top_k=top_k)
    return [h["chunk"].metadata.get("page") for h in reranked]

def page_hit(retrieved_pages, gold_pages):
    return int(any(p in gold_pages for p in retrieved_pages))

In [ ]:
rows = []
for _, row in eval_df.iterrows():
    q = row["query"]
    gold_pages = row["gold_pages"]

    # Retrieval
    base_ret = retrieve_pages(q, method="baseline", top_k=5)
    enh_ret = retrieve_pages(q, method="enhanced", top_k=5)

    rows.append({
        "query": q,
        "baseline_hit": page_hit(base_ret, gold_pages),
        "enhanced_hit": page_hit(enh_ret, gold_pages),
        "baseline_retrieved_pages": base_ret,
        "enhanced_retrieved_pages": enh_ret,
        "gold_pages": gold_pages,

        # FIXED: Replaced 'basepages' with 'base_ret' and 'row.goldpages' with 'gold_pages'
        "baseline_precision_at_5": len(set(base_ret) & set(gold_pages)) / 5,
        "enhanced_precision_at_5": len(set(enh_ret) & set(gold_pages)) / 5,
        "baseline_recall_proxy": len(set(base_ret) & set(gold_pages)) / len(set(gold_pages)),
        "enhanced_recall_proxy": len(set(enh_ret) & set(gold_pages)) / len(set(gold_pages)),
    })

# FIXED: Standardized dataframe name and fixed EVAL_DIR constant
retrieval_eval_df = pd.DataFrame(rows)
retrieval_eval_df.to_csv(os.path.join(EVAL_DIR, "retrieval_eval.csv"), index=False)
print(retrieval_eval_df.mean(numeric_only=True))

In [ ]:
import logging

logging.basicConfig(level=logging.INFO)

def safe_generate(func, query, rpm_delay=20):
    """
    Wraps generation calls with an explicit pause to respect free-tier RPM limits.
    """
    try:
        # Execute the pipeline function
        result = func(query)

        # Pause after the API call to respect the free tier rate limits
        # e.g., a 20-second delay ensures you stay under 3 Requests Per Minute (RPM)
        time.sleep(rpm_delay)
        return result
    except Exception as e:
        logging.error(f"Generation failed for query '{query}': {e}")
        time.sleep(rpm_delay) # Still pause to avoid hammering the API after an error
        return {"answer": None, "retrieved": []}

def run_free_tier_evaluation(df, sample_size=5, rpm_delay=20, output_dir="eval_results"):
    """
    Runs the evaluation on a small sample of the dataframe with built-in pacing.
    """
    os.makedirs(output_dir, exist_ok=True)

    # Safety Check: Downsample the data to prevent accidental massive runs
    if len(df) > sample_size:
        print(f"Downsampling evaluation dataset from {len(df)} to a safe sample of {sample_size} rows.")
        eval_df = df.sample(n=sample_size, random_state=42)
    else:
        eval_df = df

    results = []

    for row in eval_df.itertuples(index=False):
        query = row.query
        gold = row.gold_answer

        print(f"Evaluating query: {query[:50]}...")

        # 1. Generate baseline answer (Includes pacing delay)
        base_res = safe_generate(baseline_answer, query, rpm_delay=rpm_delay)
        base_ans = base_res.get("answer")

        # 2. Generate enhanced answer (Includes pacing delay)
        enh_res = safe_generate(enhanced_answer, query, rpm_delay=rpm_delay)
        enh_ans = enh_res.get("answer")

        # 3. Call your LLM Judge (Includes pacing delay if it uses an API)
        # For an ultra-safe free tier setup, you could use the heuristic score_answer here
        # to keep API usage down, or run llm_as_a_judge with another delay.
        # base_scores = llm_as_a_judge(query, base_ans, gold, base_res.get("retrieved", []))
        # time.sleep(rpm_delay)

        # Compiling basic structural data
        results.append({
            "query": query,
            "baseline_answer": base_ans,
            "enhanced_answer": enh_ans,
        })

    # Convert and save intermediate progress
    results_df = pd.DataFrame(results)
    results_df.to_csv(os.path.join(output_dir, "free_tier_eval.csv"), index=False)
    return results_df

In [ ]:
q = "What obligations apply to providers of GPAI models?"
try:
    out = enhanced_answer(q)
    print(out["answer"])
except Exception as e:
    print("Error:", e)